# 모듈 5: 오차역전파(Backpropagation) 수동 vs 자동 증명 실습

이 노트북은 인공신경망 커리큘럼의 **킬러 콘텐츠**입니다.

1. 교안에서 유도한 연쇄 법칙(Chain Rule) 수식을 파이토치 텐서 연산으로 **한 땀 한 땀** 구현합니다.
2. 파이토치 `.backward()` 자동 미분 엔진의 결과와 비교하여 **완벽 일치**를 증명합니다.
3. 이로써 "블랙박스"였던 역전파 내부를 완전히 해부합니다.

In [ ]:
import torch

torch.manual_seed(42)

## 1. 네트워크 및 데이터 정의 (1-1-1 구조)

가장 이해하기 쉬운 **입력 1개 → 은닉 1개 → 출력 1개** 구조를 사용합니다.

```
 x ──w₁──→ [z₁ = w₁x + b₁] ──σ──→ h ──w₂──→ [z₂ = w₂h + b₂] ──σ──→ ŷ ──→ L
```

In [ ]:
# 입력과 정답
x = torch.tensor([1.0])
y_target = torch.tensor([0.0])

# 가중치 및 편향 (requires_grad=True로 기울기 추적 활성화)
w1 = torch.tensor([0.5], requires_grad=True)
b1 = torch.tensor([0.1], requires_grad=True)
w2 = torch.tensor([0.8], requires_grad=True)
b2 = torch.tensor([-0.2], requires_grad=True)

print("=== 초기 파라미터 ===")
print(f"w1={w1.item()}, b1={b1.item()}, w2={w2.item()}, b2={b2.item()}")
print(f"입력 x={x.item()}, 정답 y={y_target.item()}")

## 2. 순전파 실행 (Forward Pass)

모듈 2에서 배운 단계별 계산 흐름을 실행하여 예측값과 손실을 구합니다.

In [ ]:
# ============ 순전파 ============
# ① 은닉층 선형결합
z1 = w1 * x + b1
print(f"① z1 = w1*x + b1 = {w1.item()}*{x.item()} + {b1.item()} = {z1.item():.4f}")

# ② 은닉층 활성화 (시그모이드)
h = torch.sigmoid(z1)
print(f"② h  = σ(z1) = σ({z1.item():.4f}) = {h.item():.4f}")

# ③ 출력층 선형결합
z2 = w2 * h + b2
print(f"③ z2 = w2*h + b2 = {w2.item()}*{h.item():.4f} + {b2.item()} = {z2.item():.4f}")

# ④ 출력 활성화
y_hat = torch.sigmoid(z2)
print(f"④ ŷ  = σ(z2) = σ({z2.item():.4f}) = {y_hat.item():.4f}")

# ⑤ 손실 계산 (간소화된 MSE: (ŷ - y)²)
loss = (y_hat - y_target) ** 2
print(f"⑤ L  = (ŷ - y)² = ({y_hat.item():.4f} - {y_target.item()})² = {loss.item():.6f}")

## 3. ⭐ 수동 역전파 (Manual Backpropagation)

교안에서 유도한 수식을 그대로 코드로 옮깁니다.
연쇄 법칙: "끊고 → 미분하고 → 곱하기" 패턴을 반복합니다.

In [ ]:
# ============ 수동 역전파 (교안 수식 그대로 코드화) ============
# 순전파에서 계산된 값들을 숫자로 저장 (그래프 추적 없이)
z1_val = z1.detach()
h_val = h.detach()
z2_val = z2.detach()
y_hat_val = y_hat.detach()

print("=" * 60)
print("수동 역전파: 연쇄 법칙 조각 미분 실행")
print("=" * 60)

# --- 조각 1: ∂L/∂ŷ ---
dL_dy_hat = 2 * (y_hat_val - y_target)
print(f"\n조각1: ∂L/∂ŷ = 2(ŷ - y) = 2({y_hat_val.item():.4f} - {y_target.item()}) = {dL_dy_hat.item():.4f}")

# --- 조각 2: ∂ŷ/∂z₂ (시그모이드 미분) ---
dy_hat_dz2 = torch.sigmoid(z2_val) * (1 - torch.sigmoid(z2_val))
print(f"조각2: ∂ŷ/∂z₂ = σ(z₂)(1-σ(z₂)) = {dy_hat_dz2.item():.4f}")

# --- 조각 3: ∂z₂/∂w₂ ---
dz2_dw2 = h_val
print(f"조각3: ∂z₂/∂w₂ = h = {dz2_dw2.item():.4f}")

# --- 조각 4: ∂z₂/∂h ---
dz2_dh = w2.detach()
print(f"조각4: ∂z₂/∂h = w₂ = {dz2_dh.item():.4f}")

# --- 조각 5: ∂h/∂z₁ (시그모이드 미분) ---
dh_dz1 = torch.sigmoid(z1_val) * (1 - torch.sigmoid(z1_val))
print(f"조각5: ∂h/∂z₁ = σ(z₁)(1-σ(z₁)) = {dh_dz1.item():.4f}")

# --- 조각 6: ∂z₁/∂w₁ ---
dz1_dw1 = x
print(f"조각6: ∂z₁/∂w₁ = x = {dz1_dw1.item():.4f}")

# --- 조각 7: ∂z₂/∂b₂ ---
dz2_db2 = torch.tensor([1.0])
print(f"조각7: ∂z₂/∂b₂ = 1")

# --- 조각 8: ∂z₁/∂b₁ ---
dz1_db1 = torch.tensor([1.0])
print(f"조각8: ∂z₁/∂b₁ = 1")

In [ ]:
# ============ 조각들을 연쇄 법칙으로 곱하여 최종 기울기 산출 ============
print("\n" + "=" * 60)
print("연쇄 법칙 적용: 조각들을 곱하여 최종 기울기 계산")
print("=" * 60)

# --- w2의 기울기: ∂L/∂w₂ = (조각1) × (조각2) × (조각3) ---
manual_grad_w2 = dL_dy_hat * dy_hat_dz2 * dz2_dw2
print(f"\n∂L/∂w₂ = ∂L/∂ŷ × ∂ŷ/∂z₂ × ∂z₂/∂w₂")
print(f"       = {dL_dy_hat.item():.4f} × {dy_hat_dz2.item():.4f} × {dz2_dw2.item():.4f}")
print(f"       = {manual_grad_w2.item():.6f}")

# --- b2의 기울기: ∂L/∂b₂ = (조각1) × (조각2) × (조각7) ---
manual_grad_b2 = dL_dy_hat * dy_hat_dz2 * dz2_db2
print(f"\n∂L/∂b₂ = ∂L/∂ŷ × ∂ŷ/∂z₂ × ∂z₂/∂b₂")
print(f"       = {dL_dy_hat.item():.4f} × {dy_hat_dz2.item():.4f} × 1")
print(f"       = {manual_grad_b2.item():.6f}")

# --- w1의 기울기 (연쇄가 길다!): ∂L/∂w₁ = (조각1) × (조각2) × (조각4) × (조각5) × (조각6) ---
manual_grad_w1 = dL_dy_hat * dy_hat_dz2 * dz2_dh * dh_dz1 * dz1_dw1
print(f"\n∂L/∂w₁ = ∂L/∂ŷ × ∂ŷ/∂z₂ × ∂z₂/∂h × ∂h/∂z₁ × ∂z₁/∂w₁")
print(f"       = {dL_dy_hat.item():.4f} × {dy_hat_dz2.item():.4f} × {dz2_dh.item():.4f} × {dh_dz1.item():.4f} × {dz1_dw1.item():.4f}")
print(f"       = {manual_grad_w1.item():.6f}")

# --- b1의 기울기: ∂L/∂b₁ = (조각1) × (조각2) × (조각4) × (조각5) × (조각8) ---
manual_grad_b1 = dL_dy_hat * dy_hat_dz2 * dz2_dh * dh_dz1 * dz1_db1
print(f"\n∂L/∂b₁ = ∂L/∂ŷ × ∂ŷ/∂z₂ × ∂z₂/∂h × ∂h/∂z₁ × ∂z₁/∂b₁")
print(f"       = {manual_grad_w1.item():.6f}와 동일 (x=1이므로)")
print(f"       = {manual_grad_b1.item():.6f}")

## 4. ⭐⭐ 자동 미분 vs 수동 미분 완전 일치 증명

파이토치의 `.backward()`가 내부적으로 수행하는 연산이
우리가 손으로 유도한 연쇄 법칙 수식과 100% 동일함을 입증합니다.

In [ ]:
# ============ PyTorch 자동 역전파 실행 ============
loss.backward()  # 이 한 줄이 위의 모든 연쇄 법칙 계산을 자동으로 수행!

print("=" * 65)
print("⭐ 최종 증명: 수동 계산 vs PyTorch 자동 미분(.backward()) 비교")
print("=" * 65)
print(f"{'파라미터':>8} | {'수동 계산':>14} | {'PyTorch 자동':>14} | {'일치 여부':>8}")
print("-" * 55)

params = [
    ('w1', manual_grad_w1, w1.grad),
    ('b1', manual_grad_b1, b1.grad),
    ('w2', manual_grad_w2, w2.grad),
    ('b2', manual_grad_b2, b2.grad),
]

all_match = True
for name, manual, auto in params:
    match = torch.allclose(manual, auto, atol=1e-6)
    all_match = all_match and match
    status = "일치" if match else "불일치"
    print(f"{name:>8} | {manual.item():>14.8f} | {auto.item():>14.8f} | {status:>8}")

print("\n" + "=" * 65)
if all_match:
    print("결과: 모든 파라미터의 기울기가 완벽하게 일치합니다.")
    print("   → 파이토치의 .backward()는 우리가 수식으로 유도한")
    print("     연쇄 법칙(Chain Rule)을 정확히 실행하는 계산 엔진입니다.")
    print("   → 신경망의 학습 과정은 더 이상 블랙박스가 아닙니다.")
else:
    print("불일치 발견 - 수식 검토가 필요합니다.")

## 5. 공식 적용 예제: 체인룰로 기울기 해석

역전파 핵심 공식(체인룰) $rac{\partial L}{\partial w}=rac{\partial L}{\partial \hat{y}}rac{\partial \hat{y}}{\partial z}rac{\partial z}{\partial w}$ 를 확인합니다.

여기서는 시그모이드 뉴런 1개에서 수동 계산과 autograd를 비교합니다.


In [ ]:
import torch

x = torch.tensor([2.0])
y_true = torch.tensor([1.0])
w = torch.tensor([0.3], requires_grad=True)
b = torch.tensor([-0.1], requires_grad=True)

z = w * x + b
y_hat = torch.sigmoid(z)
loss = 0.5 * (y_hat - y_true) ** 2
loss.backward()

# 수동 체인룰
dL_dy = (y_hat - y_true)
dy_dz = y_hat * (1 - y_hat)
dz_dw = x
manual_dw = (dL_dy * dy_dz * dz_dw).item()

print(f'autograd dL/dw = {w.grad.item():.6f}')
print(f'manual   dL/dw = {manual_dw:.6f}')
